# 🧬 المختبر 04 — PROTAC: تدمير البروتين

## Vepdegestrant (Veppanu) — أول PROTAC معتمد (FDA 2026)

### 🎯 ما ستنجزه
1. فهم آلية PROTAC (تكوين معقد ثلاثي)
2. نمذجة Hook Effect (ظاهرة فريدة)
3. مقارنة مع مثبط تقليدي (Fulvestrant)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
print("✅ جاهز")

## 📚 النظرية

PROTAC = [Ligand ERα] — [Linker] — [Ligand CRBN]

**الآلية:**
1. يرتبط بـ ERα وCRBN معًا
2. معقد ثلاثي (ERα–PROTAC–CRBN)
3. CRBN يضيف Ubiquitin
4. البروتيازوم يحلل ERα

**Hook Effect:** في التركيز العالي، يفضل ثنائيات بدلًا من ثلاثيات → أقل فعالية.

In [ ]:
def degradation(conc, dc50, dmax, kd_tern=1.0, kd_bin=100.0):
    c = np.asarray(conc, dtype=float)
    ternary = c / (kd_tern + c + (c ** 2) / kd_bin)
    t_max = ternary.max() if len(ternary) > 1 else 1
    normalized = ternary / t_max if t_max > 0 else ternary
    return dmax * normalized * (c / (dc50 + c))

def traditional(conc, dc50, dmax):
    c = np.asarray(conc, dtype=float)
    return dmax * c / (dc50 + c)

# Vepdegestrant parameters
DC50 = 0.94
DMAX = 95.0

conc = np.logspace(-2, 5, 300)
deg_vep = degradation(conc, DC50, DMAX)
deg_fulv = traditional(conc, dc50=10.0, dmax=60.0)

print(f"Vepdegestrant: DC50={DC50} nM, Dmax={DMAX}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.semilogx(conc, deg_vep, 'b-', linewidth=3, label='Vepdegestrant (PROTAC)')
ax.semilogx(conc, deg_fulv, 'r--', linewidth=2, label='Fulvestrant (inhibitor)')
ax.axvline(DC50, color='blue', linestyle=':', alpha=0.5, label=f'DC50={DC50} nM')

peak_idx = np.argmax(deg_vep)
peak_conc = conc[peak_idx]
ax.axvline(peak_conc, color='purple', linestyle=':', alpha=0.5)
ax.annotate(f'Hook peak: {peak_conc:.0f} nM', (peak_conc, deg_vep[peak_idx]),
            xytext=(50, -30), textcoords='offset points')

ax.set_xlabel('PROTAC Concentration (nM)')
ax.set_ylabel('ERα Degradation (%)')
ax.set_title('Hook Effect in Vepdegestrant')
ax.legend()
ax.set_ylim(-5, 105)
plt.show()

print(f"🔍 Hook peak: {peak_conc:.0f} nM ({peak_conc/DC50:.0f}x DC50)")

## 🎯 الدرس

**Hook Effect:** التركيز العالي ليس أفضل.
- جرعة مثالية = قمة الهوك (لا أكثر)
- Vepdegestrant: قمة ~100 nM (100x DC50)

**التطبيق السريري:**
- سرطان الثدي ER+ مع طفرة ESR1
- الجرعة: 200 mg فمويًا/يوميًا
- المصدر: FDA NDA 2026-05-01